# [cppyy](https://cppyy.readthedocs.io)

* Born from ROOT bindings (High Energy Physics, HEP)
* Built on top of Cling (a C++ interpreter)
* JIT, so it can handle templates: bindings are generated when you call them
* No build step at all, and no binding code to write

Downsides:

* Header code runs in Cling (JIT)
* Heavy *user* requirement (Cling ships a large chunk of a compiler)
* First call to anything pays the compile cost
* Historically conflicts with ROOT, which brings its own Cling

In [ ]:
import os
import subprocess
import sys

# On macOS, point Cling at the installed SDK, so it can find the system headers
if sys.platform.startswith("darwin"):
    sdk = subprocess.run(
        ["xcrun", "--show-sdk-path"], capture_output=True, text=True, check=True
    )
    os.environ.setdefault("SDKROOT", sdk.stdout.strip())

In [ ]:
%%writefile SimpleClass.hpp
#pragma once

class Simple {
    int x;
    
public:

    Simple(int x): x(x) {}
    
    int get() const {
        return x;
    }
    
};

In [ ]:
import cppyy

In [ ]:
cppyy.include("SimpleClass.hpp")
x = cppyy.gbl.Simple(5)
x.get()

In [ ]:
%%writefile SimpleTemplate.hpp
#pragma once

template<class T>
T square(T x) {
    return x*x;
}

In [ ]:
cppyy.include("SimpleTemplate.hpp")
cppyy.gbl.square(5)

In [ ]:
cppyy.gbl.square(2.5)

This is the only technology here that can bind a template *after* you have shipped the library. Pybind11 or nanobind would need a list of the overloads you want; Cython would need a fused type. In both cases you pick the types at compile time, and a user who wants `square<long double>` is out of luck. A JIT can compile it on demand.

The trade is where the compiler lives: cppyy needs one at run time, on the user's machine.

# Choosing between all of these

| You have                                        | Reach for                        |
| ----------------------------------------------- | -------------------------------- |
| A C library with a flat API                     | ctypes or CFFI                   |
| A C++ library, and you want a real package      | nanobind (or pybind11)           |
| A C++ library that is heavily templated         | cppyy                            |
| Python code that needs to be faster             | Numba first, then Cython         |
| Bindings for several languages at once          | SWIG                             |
| Rust                                            | [PyO3](https://pyo3.rs) + [maturin](https://www.maturin.rs) |

Whatever you pick, the packaging story is the same: a `pyproject.toml`, a build backend, and wheels built by [cibuildwheel](https://cibuildwheel.pypa.io). The [Scientific Python Development Guide](https://learn.scientific-python.org/development) covers that side in detail.